# T2 mask-overlay comparison: MedSAM bbox prompts and existing masks

This notebook is separate from `visualize_masks.ipynb`. It compares the original T2 slice with six masks overlaid using a yellow boundary and translucent yellow fill:

1. MedSAM centered 1/2-size bbox prompt
2. MedSAM centered 3/4-size bbox prompt
3. `T2_res_sam3_text_mask_0.6`
4. `post_train_mask`
5. `medicalsam3` generated with text prompt `prostate`
6. `medsam3` generated with text prompt `prostate`

The bbox-prompt MedSAM results are read from the sidecar H5 generated by `MedSAM/infer_h5_t2_bboxes.py`; the other masks are read from the source H5.

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np

SOURCE_H5 = Path('/data/users/lly/projects/PCa-HSD-LSDT/dataset/patients_dataset_postmask.h5')
MEDSAM_H5 = Path('/data/users/lly/projects/PCa-HSD-LSDT/dataset/patients_dataset_postmask_medsam_bbox.h5')
MASK_IMG_DIR = SOURCE_H5.parent / 'mask_img'

HALF_KEY = 'T2_medsam_bbox_half_mask'
THREE_QUARTER_KEY = 'T2_medsam_bbox_three_quarter_mask'
SAM3_KEY = 'T2_res_sam3_text_mask_0.6'
POST_KEY = 'post_train_mask'
MEDICALSAM3_KEY = 'medicalsam3'
MEDSAM3_KEY = 'medsam3'

if not SOURCE_H5.is_file():
    raise FileNotFoundError(SOURCE_H5)
if not MEDSAM_H5.is_file():
    raise FileNotFoundError(
        f'{MEDSAM_H5} does not exist. Run MedSAM/infer_h5_t2_bboxes.py first.'
    )

source = h5py.File(SOURCE_H5, 'r')
medsam = h5py.File(MEDSAM_H5, 'r')

def natural_key(value):
    return (0, int(value)) if str(value).isdigit() else (1, str(value))

patient_ids = sorted(set(source.keys()) & set(medsam.keys()), key=natural_key)
complete_patient_ids = [
    pid for pid in patient_ids
    if bool(medsam[pid].attrs.get('complete', False))
    and HALF_KEY in medsam[pid]
    and THREE_QUARTER_KEY in medsam[pid]
    and MEDICALSAM3_KEY in source[pid]
    and MEDSAM3_KEY in source[pid]
]

print(f'Source patients: {len(source)}')
print(f'Patients with complete MedSAM results: {len(complete_patient_ids)}')
print('MedSAM root metadata:')
for key, value in medsam.attrs.items():
    print(f'  {key}: {value}')

In [ ]:
def slice_2d(dataset, slice_idx):
    image = np.asarray(dataset[slice_idx])
    if image.ndim == 3 and image.shape[-1] == 1:
        image = image[..., 0]
    if image.ndim != 2:
        raise ValueError(f'Expected a 2D slice, got shape {image.shape}')
    return image

def display_limits(t2):
    finite = t2[np.isfinite(t2)]
    if finite.size == 0:
        return 0.0, 1.0
    # Preserve the stored T2 intensity range instead of percentile stretching.
    vmin, vmax = finite.min(), finite.max()
    if vmax <= vmin:
        vmax = vmin + 1.0
    return float(vmin), float(vmax)

def show_t2_with_mask(axis, t2, mask, vmin, vmax, fill_alpha=0.28):
    axis.imshow(t2, cmap='gray', vmin=vmin, vmax=vmax)
    binary_mask = mask.astype(bool)
    if binary_mask.any():
        yellow_fill = np.zeros((*binary_mask.shape, 4), dtype=np.float32)
        yellow_fill[..., :3] = (1.0, 0.84, 0.0)
        yellow_fill[..., 3] = binary_mask * fill_alpha
        axis.imshow(yellow_fill, interpolation='nearest')
        axis.contour(
            binary_mask.astype(np.uint8),
            levels=[0.5],
            colors=['#FFD400'],
            linewidths=1.4,
        )

def save_panels(
    patient_id,
    slice_idx,
    t2,
    masks,
    vmin,
    vmax,
    output_dir=MASK_IMG_DIR,
    fill_alpha=0.28,
    dpi=300,
):
    """Save the original T2 and all mask overlays as separate PNG files."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    panel_names = [
        'original_t2',
        'medsam_bbox_half',
        'medsam_bbox_three_quarter',
        'sam3_text_mask_0p6',
        'post_train_mask',
        'medicalsam3_prostate',
        'medsam3_prostate',
    ]
    saved_paths = []

    for panel_name, mask in zip(panel_names, [None, *masks]):
        panel_figure, panel_axis = plt.subplots(figsize=(4, 4))
        if mask is None:
            panel_axis.imshow(t2, cmap='gray', vmin=vmin, vmax=vmax)
        else:
            show_t2_with_mask(
                panel_axis,
                t2,
                mask,
                vmin,
                vmax,
                fill_alpha=fill_alpha,
            )
        panel_axis.axis('off')
        panel_figure.subplots_adjust(left=0, right=1, bottom=0, top=1)
        output_path = output_dir / (
            f'patient_{patient_id}_slice_{slice_idx:02d}_{panel_name}.png'
        )
        panel_figure.savefig(
            output_path,
            dpi=dpi,
            bbox_inches='tight',
            pad_inches=0,
            facecolor='white',
        )
        plt.close(panel_figure)
        saved_paths.append(output_path)

    return saved_paths

def visualize_patient(
    patient_idx=0,
    slice_idx=8,
    fill_alpha=0.28,
    save_dir=MASK_IMG_DIR,
    save_dpi=300,
):
    if not complete_patient_ids:
        raise RuntimeError('No complete MedSAM patient results are available.')
    pid = complete_patient_ids[patient_idx]
    source_group = source[pid]
    medsam_group = medsam[pid]

    num_slices = source_group['T2'].shape[0]
    if not -num_slices <= slice_idx < num_slices:
        raise IndexError(f'slice_idx={slice_idx}; valid range is 0..{num_slices - 1}')

    t2 = slice_2d(source_group['T2'], slice_idx)
    masks = [
        slice_2d(medsam_group[HALF_KEY], slice_idx),
        slice_2d(medsam_group[THREE_QUARTER_KEY], slice_idx),
        slice_2d(source_group[SAM3_KEY], slice_idx),
        slice_2d(source_group[POST_KEY], slice_idx),
        slice_2d(source_group[MEDICALSAM3_KEY], slice_idx),
        slice_2d(source_group[MEDSAM3_KEY], slice_idx),
    ]
    titles = [
        'Original T2',
        'T2 + MedSAM mask\n1/2 bbox',
        'T2 + MedSAM mask\n3/4 bbox',
        'T2 + SAM3 text mask\nthreshold 0.6',
        'T2 + Post-train mask',
        'T2 + Medical-SAM3\ntext: prostate',
        'T2 + MedSAM3\ntext: prostate',
    ]
    vmin, vmax = display_limits(t2)
    label = int(source_group['label'][()]) if 'label' in source_group else '?'

    fig, axes = plt.subplots(1, 7, figsize=(30.8, 4.8), constrained_layout=True)
    axes[0].imshow(t2, cmap='gray', vmin=vmin, vmax=vmax)
    for axis, mask in zip(axes[1:], masks):
        show_t2_with_mask(axis, t2, mask, vmin, vmax, fill_alpha=fill_alpha)
    for axis, title in zip(axes, titles):
        axis.set_title(title)
        axis.axis('off')
    fig.suptitle(f'Patient {pid} | slice {slice_idx} | label {label}', fontsize=14)
    plt.show()

    saved_paths = []
    if save_dir is not None:
        saved_paths = save_panels(
            patient_id=pid,
            slice_idx=slice_idx,
            t2=t2,
            masks=masks,
            vmin=vmin,
            vmax=vmax,
            output_dir=save_dir,
            fill_alpha=fill_alpha,
            dpi=save_dpi,
        )
        print(f'Saved {len(saved_paths)} panels to {Path(save_dir).resolve()}')

    print('Mask foreground pixels:')
    for title, mask in zip(titles[1:], masks):
        print(f'  {title.replace(chr(10), " / ")}: {int(mask.astype(bool).sum()):,}')
    print('Prompt boxes [x0, y0, x1, y1]:')
    print('  1/2:', medsam_group[HALF_KEY].attrs.get('bbox_xyxy'))
    print('  3/4:', medsam_group[THREE_QUARTER_KEY].attrs.get('bbox_xyxy'))
    return fig, saved_paths

# Change patient_idx and slice_idx as needed.
visualize_patient(patient_idx=8, slice_idx=3)

In [ ]:
# Optional overview: one middle slice from several completed patients.
def visualize_examples(patient_indices=(0, 1, 2), slice_idx=8):
    for patient_idx in patient_indices:
        visualize_patient(patient_idx=patient_idx, slice_idx=slice_idx)

# visualize_examples(patient_indices=(0, 20, 100), slice_idx=8)

In [ ]:
# Run this cell when finished with the notebook.
source.close()
medsam.close()